# XGBoost：竞赛神器
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：XGBoost.py | 核心功能：梯度提升、正则化、特征重要性、早停

## 概述

XGBoost（eXtreme Gradient Boosting）是梯度提升的高效实现，加了正则化项防止过拟合。在 Kaggle 竞赛中长期霸榜，是结构化数据的首选模型之一。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, r2_score, classification_report

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
# --- 输出结果 ---
    print("[SKIP] XGBoost 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_XGB = False
    print("XGBoost 未安装,请运行: pip install xgboost\n")

## 数学原理

### 1. XGBoost 的目标函数

XGBoost 在第 $t$ 轮最小化以下目标函数（包含正则项）：

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

其中正则项：

$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2$$

$T$ 是叶子节点数，$w_j$ 是叶节点权重。

### 2. 泰勒展开近似

对损失函数在 $\hat{y}_i^{(t-1)}$ 处二阶泰勒展开：

$$L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) \approx L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2}h_i f_t(x_i)^2$$

其中一阶和二阶梯度统计量：

$$g_i = \frac{\partial L(y_i, \hat{y}_i^{(t-1)})}{\partial \hat{y}_i^{(t-1)}}, \quad h_i = \frac{\partial^2 L(y_i, \hat{y}_i^{(t-1)})}{\partial (\hat{y}_i^{(t-1)})^2}$$

### 3. 最优叶节点权重

对每个叶节点 $j$，最优权重为：

$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

其中 $I_j$ 是落入叶节点 $j$ 的样本集合。

### 4. 最优增益

分裂一个叶节点的增益：

$$\text{Gain} = \frac{1}{2}\left[\frac{(\sum_{i \in I_L} g_i)^2}{\sum_{i \in I_L} h_i + \lambda} + \frac{(\sum_{i \in I_R} g_i)^2}{\sum_{i \in I_R} h_i + \lambda} - \frac{(\sum_{i \in I} g_i)^2}{\sum_{i \in I} h_i + \lambda}\right] - \gamma$$

- $\gamma$ 控制分裂的门槛，自动实现剪枝
- Gain $< 0$ 时不进行分裂

### 5. XGBoost 的关键优化

| 优化技术 | 说明 |
|----------|------|
| 列采样（`colsample_bytree`） | 每棵树只用部分特征，类似随机森林 |
| 行采样（`subsample`） | 每棵树只用部分样本 |
| 加权分位数（Weighted Quantile Sketch） | 高效寻找近似分裂点 |
| 缓存感知访问 | 减少缓存未命中 |
| 稀疏感知 | 自动处理缺失值 |

### 6. 与标准 Gradient Boosting 的区别

| 方面 | Gradient Boosting | XGBoost |
|------|-------------------|---------|
| 梯度 | 仅用一阶梯度 $g_i$ | 用一阶 $g_i$ + 二阶 $h_i$ |
| 正则化 | 无显式正则 | $\gamma T + \frac{1}{2}\lambda\|w\|^2$ |
| 树结构 | 贪心分裂 | 带增益剪枝 |
| 并行化 | 串行 | 特征级别并行 |

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `max_depth=3` | 树的最大深度，控制模型复杂度 |
| `learning_rate=0.1` | 步长收缩 $\nu$，$F_t = F_{t-1} + \nu f_t$ |
| `subsample=0.8` | 行采样，每轮用 $80\%$ 样本 |
| `colsample_bytree=0.8` | 列采样，每棵树用 $80\%$ 特征 |
| `eval_metric="logloss"` | 对数损失 $L = -[y\log p + (1-y)\log(1-p)]$ |
| `feature_importances_` | 基于增益（gain）的特征重要性 |
| `GridSearchCV` | 超参数搜索：$\max\_depth, \nu, M$ |

### 1. 分类任务

在分类任务上训练模型并评估性能。

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 基础 XGBoost 分类

在分类任务上训练模型并评估性能。

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
# --- 赋值/配置 ---
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False,
)
# --- 训练模型 ---
xgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

print("=== XGBoost 分类 ===")
print(f"训练准确率: {xgb_clf.score(X_train, y_train):.4f}")
print(f"测试准确率: {xgb_clf.score(X_test, y_test):.4f}")

### 3. 特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
print("\n=== 特征重要性 ===")
importances = xgb_clf.feature_importances_
for i in np.argsort(importances)[::-1]:
    print(f"  特征{i}: {importances[i]:.4f}")

### 4. 超参数调优

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== GridSearchCV ===")
param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.3],
    "n_estimators": [50, 100, 200],
# --- 继续 ---
}
gs = GridSearchCV(
    xgb.XGBClassifier(eval_metric="logloss", use_label_encoder=False, random_state=42),
    param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
# --- 训练模型 ---
gs.fit(X_train, y_train)
print(f"最佳参数: {gs.best_params_}")
print(f"最佳 CV 准确率: {gs.best_score_:.4f}")
print(f"测试准确率: {gs.best_estimator_.score(X_test, y_test):.4f}")

### 5. early_stopping

运行 5. early_stopping 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Early Stopping ===")
xgb_es = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    eval_metric="logloss", use_label_encoder=False, random_state=42,
)
# --- 训练模型 ---
xgb_es.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           early_stopping_rounds=20, verbose=False)
print(f"最优轮数: {xgb_es.best_iteration}")
print(f"测试准确率: {xgb_es.score(X_test, y_test):.4f}")

### 6. 回归任务

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== XGBoost 回归 ===")
X_r, y_r = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

xgb_reg = xgb.XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42,
)
xgb_reg.fit(X_tr, y_tr)
print(f"R²: {xgb_reg.score(X_te, y_te):.4f}")

### 7. XGBoost 核心特性

运行 7. XGBoost 核心特性 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== XGBoost 核心特性 ===")
print("1. 正则化: 目标函数包含 L1/L2 正则项,防止过拟合")
print("2. 二阶泰勒展开: 同时利用一阶梯度和二阶梯度（Hessian）,优化更精确")
print("3. 列采样: 每棵树随机选择部分特征（类似随机森林）")
print("4. 缺失值处理: 自动学习缺失值的最优分裂方向")
# --- 输出结果 ---
print("5. 并行化: 特征级别并行（不是树级别）")
print("6. 缓存感知: 按列块组织数据,优化 CPU 缓存命中率")

print("\n=== XGBoost vs sklearn GradientBoosting ===")
print("- XGBoost 更快（C++ 实现 + 并行化）")
print("- XGBoost 有内置正则化")
print("- XGBoost 支持缺失值处理")
print("- XGBoost 有 early_stopping 和自定义评估指标")

print("\n=== XGBoost 要点 ===")
print("- max_depth: 通常 3~10,越大越容易过拟合")
print("- learning_rate: 通常 0.01~0.3,与 n_estimators 互补")
print("- subsample / colsample_bytree: 引入随机性防过拟合")
print("- min_child_weight: 控制叶节点最小样本权重")

## 关键代码解释

```python
import xgboost as xgb
model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                           subsample=0.8, colsample_bytree=0.8, eval_metric="logloss")
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=10)
```

## 注意事项

1. **早停**：监控验证集性能，连续 N 轮不提升就停止
2. **学习率与树数反向调节**：lr 小则 n_estimators 需要大
3. **GPU 加速**：`tree_method="hist"` + `device="cuda"`

## 总结与延伸

以上代码展示了 **XGBoost** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **XGBoost 1.0+**：支持原生分类特征
- **DART**：在 XGBoost 中加入 Dropout
- **XGBoost vs LightGBM**：后者通常更快，但 XGBoost 更稳定
